# Sample from an LSDJ generative model

### Setup

In [1]:
%%capture
!git clone https://github.com/exilefaker/pe-lsdj.git

%cd pe-lsdj
%pip install -e .

In [4]:
import sys
sys.path.append('/content/pe-lsdj/src')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/'
DATA_FOLDER = 'LSDJ_train'
OUTPUT_FOLDER = 'LSDJ_generated'
WEIGHTS_FOLDER = 'LSDJ_weights/model_v5.1_3.4.2026'

DATA_PATH = DRIVE_PATH + DATA_FOLDER + '/orbital-final.lsdsng'
OUTPUT_PATH = DRIVE_PATH + OUTPUT_FOLDER
WEIGHTS_PATH = DRIVE_PATH + WEIGHTS_FOLDER + '/weights.eqx'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Load a model

In [ ]:
from pe_lsdj.training import load_weights
from pe_lsdj.models import LSDJTransformer
import jax.random as jr
import equinox as eqx

key = jr.PRNGKey(33)
model_key, generate_key = jr.split(key)
ref_model = LSDJTransformer(model_key, d_model=128)

model = load_weights(WEIGHTS_PATH, ref_model)

### Load a song to seed generation

In [11]:
from pe_lsdj import SongFile
from pe_lsdj.embedding import SongBanks

song_fn = DATA_PATH
sf = SongFile(song_fn)

prior_banks = SongBanks.from_songfile(sf)

# Condition on first 4 steps
input_tokens = sf.song_tokens[:64,:,:]

### Generate & save

In [12]:
from pe_lsdj.generation import generate
import equinox as eqx

NUM_STEPS = 128
gen_song_tokens, gen_banks = eqx.filter_jit(generate)(
    model, 
    input_tokens, 
    generate_key, 
    prior_banks, 
    num_steps=NUM_STEPS,
)

In [13]:
out_file = SongFile.from_tokens(
    gen_song_tokens, gen_banks, name="GEN"
)
out_file.to_lsdsng(OUTPUT_PATH + "/sample3.lsdsng")

Save to /content/drive/MyDrive/LSDJ_generated/sample3.lsdsng complete.
